# 06 · LightGBM 與 CatBoost

XGBoost 不是唯一的 boosting 庫。**LightGBM**（微軟）以速度著稱，**CatBoost**（Yandex）擅長類別特徵。這堂課比較它們，讓你知道何時該用哪個。

## 學習目標

- 用 `LGBMClassifier` 訓練，體會它的速度
- 理解 LightGBM 的 leaf-wise 生長與 XGBoost 的差異
- 知道 CatBoost 的定位（雖然本課不安裝）

## 1. 速度對決：XGBoost vs LightGBM

造一份比較大的資料，比較兩者的訓練時間與準確率。

In [1]:
import time
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

X, y = make_classification(n_samples=40000, n_features=50, n_informative=15, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

def bench(model, name):
    t = time.perf_counter()
    model.fit(X_train, y_train)
    dt = time.perf_counter() - t
    print(f"{name:<12} 訓練 {dt:5.2f}s   測試準確率 {model.score(X_test, y_test):.3f}")

bench(XGBClassifier(n_estimators=300, max_depth=6, eval_metric="logloss", random_state=42), "XGBoost")
bench(LGBMClassifier(n_estimators=300, max_depth=6, verbose=-1, random_state=42), "LightGBM")

XGBoost      訓練  1.97s   測試準確率 0.977


LightGBM     訓練  2.09s   測試準確率 0.976


/Users/isaac/Documents/workspace/wemee.github.io/notebooks/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM 通常明顯更快，準確率相近——資料越大差距越明顯。

## 2. 差在哪？level-wise vs leaf-wise

- **XGBoost（level-wise）**：一層一層長，整棵樹平衡。
- **LightGBM（leaf-wise）**：每次挑「最能降低 loss 的那片葉子」往下長，樹會比較不平衡但成長更有效率 → 更快、常更準，但小資料較容易過擬合（用 `num_leaves`、`min_child_samples` 控制）。

## 3. CatBoost 的定位

CatBoost（本課不裝，Colab 可 `pip install catboost`）的招牌是**原生處理類別特徵**——你不用自己做 One-Hot，丟字串欄位進去就行，還用「ordered boosting」減少過擬合。**資料有很多類別欄位時，CatBoost 常是首選。**

## 選用速查

| 情境 | 建議 |
| --- | --- |
| 通用、生態最成熟 | XGBoost |
| 資料大、要快 | LightGBM |
| 很多類別特徵 | CatBoost |

## 小結

- 三大 boosting 庫 API 都跟 sklearn 一致，可互換試。
- LightGBM 用 **leaf-wise** 生長，通常更快。
- CatBoost 原生吃類別特徵，省去 One-Hot。

## 練習

1. 把資料量加到 `n_samples=200000`，XGBoost 與 LightGBM 的時間差距變多大？
2. 調 LightGBM 的 `num_leaves`（如 15 vs 255），對小資料的過擬合有何影響？

下一課，學會用 **SHAP** 真正解釋模型的每一個預測。